In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
from torch.amp import GradScaler, autocast

In [3]:

torch.cuda.empty_cache()
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU Memory Allocated: 0.00 GB
GPU Memory Cached: 0.00 GB


In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from torch.nn import TransformerEncoder, TransformerEncoderLayer

# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 25
NUM_CLASSES = 3
CLASS_NAMES = ['covid', 'normal', 'pneumonia']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dataset
class ChestXRayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        
        for class_idx, class_name in enumerate(CLASS_NAMES):
            class_dir = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_dir):
                if img_name.endswith(('.jpg', '.png', '.jpeg')):
                    self.samples.append((
                        os.path.join(class_dir, img_name),
                        class_idx
                    ))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')  # Changed to RGB for consistency
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
])

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom CNN + Transformer Hybrid Model
class CustomCNNTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Enhanced CNN Backbone (4 conv blocks)
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))  # Global average pooling
        )
        
        # Transformer Head (matches ResNet50 hybrid style)
        self.transformer_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),  # Project to higher dimension
            nn.Unflatten(1, (1, 512)),  # Single token
            nn.TransformerEncoder(
                nn.TransformerEncoderLayer(
                    d_model=512,
                    nhead=8,
                    dim_feedforward=2048,
                    dropout=0.1,
                    activation='gelu',
                    batch_first=True
                ),
                num_layers=2
            ),
            nn.Flatten()
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, NUM_CLASSES)
        )
    
    def forward(self, x):
        # CNN feature extraction
        x = self.cnn(x)  # (B, 256, 1, 1)
        
        # Transformer processing
        x = self.transformer_head(x)  # (B, 512)
        
        # Classification
        return self.classifier(x)

# Metrics and Visualization
def compute_metrics(model, data_loader, criterion):
    model.eval()
    all_preds, all_labels = [], []
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            running_loss += loss.item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return {
        'loss': running_loss / len(data_loader),
        'accuracy': 100 * correct / total,
        'f1': f1_score(all_labels, all_preds, average='weighted'),
        'cm': confusion_matrix(all_labels, all_preds),
        'preds': all_preds,
        'labels': all_labels
    }

def plot_confusion_matrix(cm, class_names, filename):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.savefig(filename)
    plt.close()

# Training Loop
def train_model():
    # Initialize
    torch.cuda.empty_cache()
    scaler = GradScaler(enabled=torch.cuda.is_available())
    
    # Data
    train_dataset = ChestXRayDataset('../split_data/train', train_transform)
    test_dataset = ChestXRayDataset('../split_data/test', test_transform)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, pin_memory=True)
    
    # Model
    model = CustomCNNTransformer().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    # Training tracking
    best_f1 = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'train_f1': [], 'val_f1': []}
    
    for epoch in range(EPOCHS):
        # Training phase
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        train_preds, train_labels = [], []
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for inputs, labels in train_pbar:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            
            with autocast(enabled=torch.cuda.is_available()):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            _, preds = torch.max(outputs, 1)
            train_loss += loss.item()
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())
            
            train_pbar.set_postfix({
                'loss': f"{train_loss/(train_pbar.n+1):.4f}",
                'acc': f"{100*train_correct/train_total:.2f}%"
            })
        
        # Validation phase
        val_metrics = compute_metrics(model, test_loader, criterion)
        
        # Update history
        history['train_loss'].append(train_loss/len(train_loader))
        history['train_acc'].append(100*train_correct/train_total)
        history['train_f1'].append(f1_score(train_labels, train_preds, average='weighted'))
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_f1'].append(val_metrics['f1'])
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"Train Loss: {history['train_loss'][-1]:.4f} | Acc: {history['train_acc'][-1]:.2f}% | F1: {history['train_f1'][-1]:.4f}")
        print(f"Val Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.2f}% | F1: {val_metrics['f1']:.4f}")
        
        # Save best model
        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'f1': best_f1,
                'metrics': val_metrics
            }, 'best_custom_cnn_transformer.pth')
            
            # Save confusion matrix
            plot_confusion_matrix(val_metrics['cm'], CLASS_NAMES, f'best_model_cm_epoch{epoch+1}.png')
            
            # Save classification report
            with open('best_model_report.txt', 'w') as f:
                f.write(classification_report(val_metrics['labels'], val_metrics['preds'], 
                        target_names=CLASS_NAMES, digits=4))
            
            print("↳ New best model saved!")
        
        # LR scheduler step
        scheduler.step(val_metrics['f1'])
    
    # Plot training curves
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Val')
    plt.title('Loss Curve')
    plt.legend()
    
    plt.subplot(1, 3, 2)
    plt.plot(history['train_acc'], label='Train')
    plt.plot(history['val_acc'], label='Val')
    plt.title('Accuracy Curve')
    plt.legend()
    
    plt.subplot(1, 3, 3)
    plt.plot(history['train_f1'], label='Train')
    plt.plot(history['val_f1'], label='Val')
    plt.title('F1 Score Curve')
    plt.legend()
    
    plt.savefig('training_curves.png')
    plt.close()
    
    print(f"\nTraining complete! Best Val F1: {best_f1:.4f}")
    return history

if __name__ == "__main__":
    train_model()

C:\Users\sril\AppData\Local\Temp\ipykernel_22364\765374512.py:180: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
Epoch 1/25 [Train]:   0%|                                                                      | 0/172 [00:00<?, ?it/s]C:\Users\sril\AppData\Local\Temp\ipykernel_22364\765374512.py:210: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
Epoch 1/25 [Train]: 100%|███████████████████████████████████| 172/172 [03:02<00:00,  1.06s/it, loss=0.7006, acc=69.85%]



Epoch 1 Summary:
Train Loss: 0.7006 | Acc: 69.85% | F1: 0.6986
Val Loss: 0.7221 | Acc: 74.25% | F1: 0.7265
↳ New best model saved!


Epoch 2/25 [Train]:   0%|                                                                      | 0/172 [00:00<?, ?it/s]C:\Users\sril\AppData\Local\Temp\ipykernel_22364\765374512.py:210: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
Epoch 2/25 [Train]: 100%|███████████████████████████████████| 172/172 [02:44<00:00,  1.04it/s, loss=0.4886, acc=81.35%]



Epoch 2 Summary:
Train Loss: 0.4886 | Acc: 81.35% | F1: 0.8145
Val Loss: 0.4341 | Acc: 83.56% | F1: 0.8365
↳ New best model saved!


Epoch 3/25 [Train]:   0%|                                                                      | 0/172 [00:00<?, ?it/s]C:\Users\sril\AppData\Local\Temp\ipykernel_22364\765374512.py:210: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
Epoch 3/25 [Train]: 100%|███████████████████████████████████| 172/172 [02:39<00:00,  1.08it/s, loss=0.4075, acc=84.68%]


...


Epoch 23 Summary:
Train Loss: 0.1875 | Acc: 93.72% | F1: 0.9373
Val Loss: 0.2067 | Acc: 93.82% | F1: 0.9387


Epoch 24/25 [Train]:   0%|                                                                     | 0/172 [00:00<?, ?it/s]C:\Users\sril\AppData\Local\Temp\ipykernel_22364\765374512.py:210: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
Epoch 24/25 [Train]: 100%|██████████████████████████████████| 172/172 [02:36<00:00,  1.10it/s, loss=0.1680, acc=94.49%]



Epoch 24 Summary:
Train Loss: 0.1680 | Acc: 94.49% | F1: 0.9451
Val Loss: 0.1863 | Acc: 93.96% | F1: 0.9396


Epoch 25/25 [Train]:   0%|                                                                     | 0/172 [00:00<?, ?it/s]C:\Users\sril\AppData\Local\Temp\ipykernel_22364\765374512.py:210: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
Epoch 25/25 [Train]: 100%|██████████████████████████████████| 172/172 [02:37<00:00,  1.10it/s, loss=0.1571, acc=94.54%]



Epoch 25 Summary:
Train Loss: 0.1571 | Acc: 94.54% | F1: 0.9456
Val Loss: 0.1773 | Acc: 94.33% | F1: 0.9433
↳ New best model saved!

Training complete! Best Val F1: 0.9433
